# Prvi stik s podatki — B: vaja (telekom)
**Akademija UI za poslovne aplikacije — Modul 1, Python ekosistem in spoznavanje s surovimi podatki**

Predavatelj je v zvezku `A_prvi_stik_s_podatki` postopek pokazal na spletni trgovini.
Zdaj ga ponovite sami na podatkih telekoma.

Celico zaženete s **Shift + Enter**.

### Ponovitev
- **DataFrame** = tabela: vrstica je opazovanje, stolpec je spremenljivka. En stolpec je **Series**.
- **Vektorizacija**: operacijo zapišemo za cel stolpec naenkrat — brez `for` zanke.
- Štirje ukazi za vsak nov nabor podatkov: `pd.read_csv()`, `.shape`, `.head()`, `.info()` / `.dtypes`.
- **Tip v Pandasu ni isto kot tip po pomenu.** Pandas ne ve, da je poštna številka oznaka in da je `odhod` skupina.
  Pogoste napake: datum kot besedilo (`object` / `str`), poštna številka kot število, celo število kot `float`.
- Cilj: **ena vrstica = eno opazovanje** (stranka, naročilo, …), en stolpec = ena lastnost.

### Nastavitev okolja
Zaženite spodnjo celico (izberite jo in pritisnite **Shift + Enter**). Ničesar ne spreminjajte.
Celica deluje lokalno in v Google Colabu.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:           # Google Colab: prenesemo gradivo z GitHuba
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not KOREN.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotnine>=0.15", "kagglehub"], check=True)
else:                                       # lokalno: poiščemo mapo, v kateri je utils/orodja.py
    KOREN = Path.cwd()
    while not (KOREN / "utils/orodja.py").exists() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
sys.path.insert(0, str(KOREN))              # da Python najde utils/orodja.py
print("Gradivo je v:", KOREN)

### Podatki
Funkcija `poskrbi_za_podatke()` iz `utils/orodja.py` ob prvem zagonu prenese podatke s Kaggla (~1 minuta)
in jih shrani v mapo `data/`. Kako to deluje, smo pokazali v zvezku `A_prvi_stik_s_podatki`.

In [ ]:
from utils.orodja import poskrbi_za_podatke
DATA = poskrbi_za_podatke()

---
# Del B — Vaja: telekom
Podatki ameriškega mobilnega operaterja: **100 000 naročnikov**, za vsakega podatki o porabi, napravi, gospodinjstvu
in ali je **odšel** (`odhod = 1`). Izbrali smo 22 stolpcev in jih poimenovali po slovensko.

## B1. Naložite podatke
Zaženite celice. Ničesar ni treba spreminjati.

In [ ]:
import numpy as np
import pandas as pd

telekom = pd.read_csv(DATA / "telekom.csv")
slovar = pd.read_csv(DATA / "telekom_slovar.csv")
slovar          # kaj pomeni kateri stolpec

In [ ]:
telekom.shape

In [ ]:
telekom.head()

In [ ]:
telekom.info()

**Vprašanja** (odgovorite na glas ali si zapišite):
1. Koliko strank in koliko stolpcev imamo?
2. Kateri stolpci so tipa `object` / `str` (besedilo)? Je to smiselno?
3. Kateri stolpci so `float`, čeprav bi pričakovali cela števila? Zakaj?
4. Je vsaka stranka v tabeli samo enkrat? Namig: `telekom["id_stranke"].nunique()`

<details>
<summary><b>Rešitev (klikni)</b></summary>

1. 100 000 strank, 22 stolpcev.
2. `splet_na_napravi`, `dualband`, `regija`, `zakonski_stan` — to so kategorije, besedilo je smiselno.
3. `stevilo_telefonov`, `dohodek_razred`, `stevilo_avtomobilov`: v Pandasu (verzija 2) celoštevilski stolpec ne more vsebovati
   `NaN`, zato ga Pandas spremeni v `float`. **Float pogosto pomeni: nekaj manjka.**
   Pozor tudi: `id_stranke` in `odhod` sta številki, a z njima ne računamo.
4. Da: 100 000 različnih `id_stranke` → **ena vrstica = ena stranka**.

</details>

Preverimo trditvi iz rešitve: koliko vrednosti manjka v teh treh stolpcih in koliko je različnih strank?

In [ ]:
print(telekom[["stevilo_telefonov", "dohodek_razred", "stevilo_avtomobilov"]].isna().sum())
print("Različnih id_stranke:", telekom["id_stranke"].nunique())

## B2. Raziščite posamezne stolpce
En stolpec tabele je **Series**: `telekom["zakonski_stan"]`. Nad njim kličemo metode:
- `.dtype` — tip, `.nunique()` — število različnih vrednosti, `.isna().sum()` — število manjkajočih,
- `.value_counts(dropna=False)` — kolikokrat se pojavi katera vrednost (z `dropna=False` prešteje tudi `NaN`).

V spodnji celici **zamenjajte ime stolpca** (med narekovaji) in jo ponovno zaženite.
Poskusite npr. `"zakonski_stan"`, `"splet_na_napravi"`, `"dohodek_razred"`, `"regija"`.

In [ ]:
STOLPEC = "zakonski_stan"      # ← zamenjajte

s = telekom[STOLPEC]
print("Tip v Pandasu:      ", s.dtype)
print("Različnih vrednosti:", s.nunique())
print("Manjkajočih (NaN):  ", s.isna().sum())
s.value_counts(dropna=False).head(8)

Isto kodo bomo potrebovali večkrat, zato jo **zapakiramo v funkcijo**. `def` ustvari funkcijo,
v oklepaju so vhodni podatki (tabela in ime stolpca). Kopija funkcije je v `utils/orodja.py`,
zato jo v naslednjih zvezkih samo uvozimo: `from utils.orodja import opis_stolpca`.

In [ ]:
def opis_stolpca(df, stolpec):
    s = df[stolpec]
    print("Tip v Pandasu:      ", s.dtype)
    print("Različnih vrednosti:", s.nunique())
    print("Manjkajočih (NaN):  ", s.isna().sum())
    print(s.value_counts(dropna=False).head(8).to_string())

opis_stolpca(telekom, "dualband")

**Vprašanje:** pri `zakonski_stan` je največ vrednosti `U`. Kaj mislite, kaj pomeni? Je to res kategorija?

<details>
<summary><b>Rešitev (klikni)</b></summary>

`U` pomeni *unknown* — **neznano**. To je manjkajoča vrednost, ki se skriva kot navadna kategorija,
zato je `isna()` ne najde. Podobno `UNKW` v `splet_na_napravi` in `U` v `dualband`.
Popoldne bomo takšne vrednosti iskali in obravnavali.

</details>

Preverimo: vse vrednosti teh treh stolpcev, skupaj z `NaN`:

In [ ]:
for stolpec in ["zakonski_stan", "splet_na_napravi", "dualband"]:
    print(stolpec, telekom[stolpec].value_counts(dropna=False).to_dict())

---
### Za hitre
Zaženite `telekom.describe().T` (`.T` obrne tabelo: en stolpec = ena vrstica) in poglejte:
1. Stolpec **min**: kje so negativne vrednosti? Je to mogoče?
2. Stolpca **75%** in **max**: kje je največja vrednost zelo daleč od ostalih?
3. Stolpec **count**: zakaj nimajo vsi stolpci 100 000 vrednosti?
4. Kaj pomeni povprečje (`mean`) pri `id_stranke`? In pri `odhod`?

Primerjajte lahko tudi `telekom["regija"].value_counts()` in `telekom["regija"].value_counts(dropna=False)`.